# 01 — Batch Bronze (3 maestros CSV)

Ingesta de `customers_orgs`, `users` y `billing_monthly` desde **Landing** hacia **Bronze** con PySpark.

**Salida esperada:** Parquet tipificado en `bronze/{dataset}/` particionado por `ingest_date`, con columnas técnicas `ingest_ts` y `source_file`.

In [ ]:
# Setup (Colab o local)
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pyspark
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Ajustar según dónde copies el dataset en Drive:
    # os.environ["DATA_ROOT"] = "/content/datalake"
    PROJECT_ROOT = Path("/content/drive/MyDrive/cloud-provider-analytics")
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_ROOT, LANDING, BRONZE

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_ROOT:    {DATA_ROOT}")
print(f"LANDING:      {LANDING}")
print(f"BRONZE:       {BRONZE}")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("batch-bronze")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

In [ ]:
from src.jobs.bronze_batch import run_batch_bronze

results_run1 = run_batch_bronze(spark)
results_run1

In [ ]:
import pandas as pd

evidence_df = pd.DataFrame(results_run1)[
    [
        "dataset_name",
        "raw_count",
        "deduped_count",
        "removed_duplicates",
        "written_count",
        "bronze_path",
    ]
]
evidence_df

In [ ]:
for dataset in ["customers_orgs", "users", "billing_monthly"]:
    path = f"{BRONZE}/{dataset}"
    print(f"\n=== {dataset} ===")
    df = spark.read.parquet(path)
    df.printSchema()
    df.select(df.columns[:6]).show(5, truncate=False)

In [ ]:
from src.jobs.bronze_batch import validate_bronze_uniqueness

validations = validate_bronze_uniqueness(spark)
pd.DataFrame(validations)

In [ ]:
# Re-ejecución idempotente: conteos deduped deben mantenerse
results_run2 = run_batch_bronze(spark)

for r1, r2 in zip(results_run1, results_run2):
    assert r1["deduped_count"] == r2["deduped_count"], r1["dataset_name"]
    assert r2["written_count"] == r2["deduped_count"], r2["dataset_name"]

print("Idempotencia OK: re-ejecución sin duplicados por clave natural.")